In [1]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select, WebDriverWait
import re

load_dotenv()

# Configure optimized browser properties
chrome_options = webdriver.ChromeOptions()
chrome_options.page_load_strategy = (
    'eager'  # Speeds up interaction by omitting heavy media loading
)
driver = webdriver.Chrome(options=chrome_options)

search_url = os.getenv('PORTAL')
driver.get(search_url)
wait = WebDriverWait(driver, 10)

# -------------------------------------------------------------
# 1. Load the Proposal Numbers from the Target Excel File
# -------------------------------------------------------------
excel_path = os.getenv('ActiveLeads')

if not os.path.exists(excel_path):
  raise FileNotFoundError(
      f'Could not find the target Excel file at: {excel_path}'
  )

df_leads = pd.read_excel(excel_path)

if 'Proposal No.' not in df_leads.columns:
  raise KeyError("The Excel file must contain a column named 'Proposal No.'")

# Initialize tracking columns cleanly
for col in ['proposal details', 'Proposal URL', 'Project Details XML']:
  if col not in df_leads.columns:
    df_leads[col] = 'N/A'
  else:
    df_leads[col] = df_leads[col].astype(object).fillna('N/A')

print(
    f'Loaded {len(df_leads)} rows from Excel sheet. Starting optimized search'
    ' loop...'
)
main_window = driver.current_window_handle

# -------------------------------------------------------------
# 2. Search Loop for each Proposal Number
# -------------------------------------------------------------
for idx, row in df_leads.iterrows():
  proposal_no = str(row['Proposal No.']).strip()

  # Skip if Proposal No. is missing or invalid
  if (
      pd.isna(row['Proposal No.'])
      or proposal_no == ''
      or proposal_no.lower() == 'nan'
  ):
    continue

  # --- FILTER CONDITION ---
  # Check if 'proposal details' is already populated
  prop_details_val = str(row['proposal details']).strip()
  if (
      not pd.isna(row['proposal details'])
      and prop_details_val not in ['', 'N/A', 'nan', 'NaN']
  ):
    print(
        f'[{idx + 1}/{len(df_leads)}] Skipping Proposal: {proposal_no} (Already'
        ' processed)'
    )
    continue

  print(f'\n[{idx + 1}/{len(df_leads)}] Processing Proposal: {proposal_no}')

  try:
    if (
        'trackYourProposal' not in driver.current_url
        or 'proposal-details' in driver.current_url
    ):
      driver.get(search_url)

    proposal_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@formcontrolname='proposalNumber']")
        )
    )
    proposal_input.clear()
    proposal_input.send_keys(proposal_no)

    search_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[@type='submit' and contains(.,'Search')]")
        )
    )
    driver.execute_script('arguments[0].click();', search_button)

    try:
      WebDriverWait(driver, 7).until(
          EC.text_to_be_present_in_element(
              (By.XPATH, "//table[@id='excel-table']/tbody/tr[1]/td[2]"),
              proposal_no,
          )
      )
    except TimeoutException:
      print(
          '  ⚠️ No matching records found or table timed out updating for:'
          f' {proposal_no}. Skipping...'
      )
      continue

    proposal_link = driver.find_element(
        By.XPATH, "//table[@id='excel-table']/tbody/tr[1]/td[2]/a"
    )

    current_handles_count = len(driver.window_handles)
    driver.execute_script('arguments[0].click();', proposal_link)

    try:
      wait.until(
          lambda d: len(d.window_handles) > current_handles_count
          or 'proposal-details' in d.current_url
      )
    except TimeoutException:
      pass

    details_url = 'N/A'
    view_proposal_url = 'N/A'
    project_details_xml = 'N/A'

    opened_in_new_tab = len(driver.window_handles) > 1
    if opened_in_new_tab:
      details_window = [
          w for w in driver.window_handles if w != main_window
      ][0]
      driver.switch_to.window(details_window)

    details_url = driver.current_url
    print(f'  -> Captured Details URL: {details_url}')

    # -------------------------------------------------------------
    # 3. Inside Details Page: Open & Wait for "View Proposal" Content
    # -------------------------------------------------------------
    try:
      view_proposal_element = wait.until(
          EC.visibility_of_element_located((
              By.XPATH,
              "//a[contains(@class, 'btn') and contains(text(), 'View"
              " Proposal')]",
          ))
      )

      time.sleep(1.5)
      pre_click_windows = driver.window_handles

      driver.execute_script('arguments[0].click();', view_proposal_element)

      print('  -> Waiting for Project Details container to settle...')
      time.sleep(3.5)

      try:
        container_element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//*[@id='caf_project'] | //*[contains(@id,'caf_project')]",
            ))
        )
        project_details_xml = container_element.get_attribute('outerHTML')
        print(
            '  -> Captured Project Details Container XML successfully.'
        )
      except TimeoutException:
        try:
          table_element = driver.find_element(
              By.XPATH,
              "//h4[contains(text(),'Project Details')]/following::table[1] |"
              ' //table[1]',
          )
          project_details_xml = table_element.get_attribute('outerHTML')
          print(
              '  -> Captured Project Details Table XML via structure layout'
              ' fallback.'
          )
        except Exception:
          print(
              '  ⚠️ Could not isolate the project details layout tree'
              ' structure.'
          )

      post_click_windows = driver.window_handles
      if len(post_click_windows) > len(pre_click_windows):
        proposal_doc_window = [
            w for w in post_click_windows if w not in pre_click_windows
        ][0]
        driver.switch_to.window(proposal_doc_window)
        view_proposal_url = driver.current_url
        driver.close()
        if opened_in_new_tab:
          driver.switch_to.window(details_window)
        else:
          driver.switch_to.window(main_window)
      else:
        view_proposal_url = driver.current_url

      print(f'  -> Captured View Proposal URL: {view_proposal_url}')

    except (NoSuchElementException, TimeoutException) as err:
      print(
          "  ⚠️ 'View Proposal' link interaction sequence encountered an"
          f' exception: {err}'
      )

    # -------------------------------------------------------------
    # 4. Clean Master Reset to Search Index Screen
    # -------------------------------------------------------------
    if opened_in_new_tab:
      driver.close()
      driver.switch_to.window(main_window)

    driver.get(search_url)
    wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@formcontrolname='proposalNumber']")
        )
    )

    df_leads.at[idx, 'proposal details'] = details_url
    df_leads.at[idx, 'Proposal URL'] = view_proposal_url
    df_leads.at[idx, 'Project Details XML'] = project_details_xml

  except Exception as e:
    print(f' ❌ Global processing fail for proposal {proposal_no}: {e}')
    all_windows = driver.window_handles
    if len(all_windows) > 1:
      for extra_w in all_windows[1:]:
        driver.switch_to.window(extra_w)
        driver.close()
    driver.switch_to.window(main_window)
    driver.get(search_url)
    continue

# -------------------------------------------------------------
# 5. Post-Loop Sheet Save Back Execution
# -------------------------------------------------------------
print(
    '\n[Final Step] Saving all collected data and URLs to the Excel file in a'
    ' single batch...'
)
# Remove non-printable ASCII control characters (keeping standard newlines/tabs)
ILLEGAL_CHARACTERS_RE = re.compile(
    r"[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F]"
)


def clean_illegal_chars(val):
    if isinstance(val, str):
        return ILLEGAL_CHARACTERS_RE.sub("", val)
    return val


# Apply cleaning to the entire dataframe
df_leads_clean = df_leads.map(
    lambda x: clean_illegal_chars(x)
)  # or df_leads.map(clean_illegal_chars) in pandas >= 2.1

# Export to Excel
df_leads_clean.to_excel(excel_path, index=False)

print(f'🎉 Task complete! All data successfully saved to: {excel_path}')

driver.quit()

Loaded 5471 rows from Excel sheet. Starting optimized search loop...
[1/5471] Skipping Proposal: IA/GJ/IND2/429460/2023 (Already processed)
[2/5471] Skipping Proposal: SIA/RJ/IND1/466442/2024 (Already processed)
[3/5471] Skipping Proposal: IA/GJ/IND3/547415/2025 (Already processed)
[4/5471] Skipping Proposal: SIA/PB/IND1/430223/2023 (Already processed)
[5/5471] Skipping Proposal: SIA/GJ/IND3/552750/2025 (Already processed)
[6/5471] Skipping Proposal: SIA/GJ/IND3/487190/2024 (Already processed)
[7/5471] Skipping Proposal: SIA/GJ/IND3/406128/2022 (Already processed)
[8/5471] Skipping Proposal: SIA/GJ/IND3/465723/2024 (Already processed)
[9/5471] Skipping Proposal: SIA/OR/IND1/525643/2025 (Already processed)
[10/5471] Skipping Proposal: IA/GJ/IND3/410132/2023 (Already processed)
[11/5471] Skipping Proposal: SIA/AS/IND2/561534/2025 (Already processed)
[12/5471] Skipping Proposal: SIA/AS/IND2/579610/2026 (Already processed)
[13/5471] Skipping Proposal: IA/MH/IND3/560966/2026 (Already proces